# Optimise coeffs from DEAP output

Try to improve the existing best coefficients from the genetic algorithm results by running them through optimisation functions.

Try `scipy.optimize.minimize` and simuleated annealing.

__Inputs:__

+ MSOA-level admissions numbers, numbers of people in each age band, and deprivation quantile.
+ SSNAP-derived stroke admission coefficients (probability of stroke given age band).
+ Admission numbers in England by age band.

__Results:__

+ Data: optimised coefficients for the five top fits from the genetic algorithm.
  + File name: `post_deap_multi_optimise_coeffs_results.csv`
+ Data: optimised coefficients for all of the genetic algorithm output from simulated annealing (basin hopping).
  + File name: `post_deap_multi_basinhop_coeffs_results.csv`

__Method:__

Convert the genetic algorithm output from scale factors to the probability of stroke admission by multiplying them by the starting SSNAP coefficients.

First try just a straight minimisation with the five best results from the genetic algorithm, but see that the resulting coefficients don't change much so not a good method for this case - doubt we have bang on the best values to start with, expect them to change quite a bit.

Then try simulated annealing with all of the best outputs from the genetic algorithm.

This notebook just runs the optimisations. The results are checked in the next notebook.

In [1]:
import os
import polars as pl
from scipy.optimize import minimize, basinhopping
import numpy as np
import matplotlib.pyplot as plt

## Load data

Patient demographics by MSOA:

In [2]:
path_to_msoa_stats = os.path.join('data', 'msoa_cleaned.csv')

df_stats = pl.read_csv(path_to_msoa_stats)

In [3]:
df_stats.head()

MSOA,admissions,IMD2019Score,All persons,country,good_health,fair health,bad health,prop_good_health,prop_fair health,prop_bad health,MSOA11CD,age_65_proportion,age_70_proportion,age_75_proportion,age_less65_proportion,age_over80_proportion,total_health,age_less65,age_65,age_70,age_75,age_over80,depriv_quantile_min,depriv_quantile_max
str,f64,f64,i64,str,i64,i64,i64,f64,f64,f64,str,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64
"""Adur 001""",14.333333,16.924833,8815,"""E""",6799,1251,474,0.79763,0.146762,0.055608,"""E02006534""",0.0559,0.0528,0.0422,0.7872,0.062,8524,6939.168,492.7585,465.432,371.993,546.53,0.4,0.6
"""Adur 002""",7.333333,6.4704,7263,"""E""",5537,838,259,0.83464,0.126319,0.039041,"""E02006535""",0.0578,0.0774,0.0492,0.7467,0.0692,6634,5423.2821,419.8014,562.1562,357.3396,502.5996,0.8,1.0
"""Adur 003""",9.333333,13.7334,7354,"""E""",5820,969,311,0.819718,0.136479,0.043803,"""E02006536""",0.0609,0.0582,0.0421,0.7729,0.0661,7100,5683.9066,447.8586,428.0028,309.6034,486.0994,0.6,0.8
"""Adur 004""",21.0,26.199857,10582,"""E""",7872,1546,709,0.777328,0.152661,0.070011,"""E02006537""",0.0465,0.0438,0.0367,0.8091,0.0638,10127,8561.8962,492.063,463.4916,388.3594,675.1316,0.2,0.4
"""Adur 005""",13.666667,11.7948,9059,"""E""",7106,1081,339,0.833451,0.126789,0.039761,"""E02006538""",0.0597,0.067,0.0425,0.7643,0.0662,8526,6923.7937,540.8223,606.953,385.0075,599.7058,0.6,0.8


Pick out column names for the health and age proportions:

In [4]:
health_numbers = ['good_health', 'fair health', 'bad health']
props_health = ['prop_good_health', 'prop_fair health', 'prop_bad health']
props_age = [
    'age_less65_proportion', 'age_65_proportion', 'age_70_proportion',
    'age_75_proportion', 'age_over80_proportion'
]
age_numbers = [p.replace('_proportion', '') for p in props_age]

In [5]:
qmin_list = sorted(df_stats['depriv_quantile_min'].unique())

In [6]:
# Names of coeffs:
coeff_names = [f'{a}_q{str(round(q, 1)).replace(".", "")}' for q in qmin_list for a in age_numbers]

coeff_names

['age_less65_q00',
 'age_65_q00',
 'age_70_q00',
 'age_75_q00',
 'age_over80_q00',
 'age_less65_q02',
 'age_65_q02',
 'age_70_q02',
 'age_75_q02',
 'age_over80_q02',
 'age_less65_q04',
 'age_65_q04',
 'age_70_q04',
 'age_75_q04',
 'age_over80_q04',
 'age_less65_q06',
 'age_65_q06',
 'age_70_q06',
 'age_75_q06',
 'age_over80_q06',
 'age_less65_q08',
 'age_65_q08',
 'age_70_q08',
 'age_75_q08',
 'age_over80_q08']

Pick out data for calculating admissions:

In [7]:
all_x_lists = []
all_admissions = []

for qmin in qmin_list:
    mask = (df_stats['depriv_quantile_min'] == qmin)
    # Keep only those MSOA:
    df_stats_here = df_stats.filter(mask)
    # MSOA data in the same order as those coefficients:
    x_lists = [df_stats_here[a] for a in age_numbers]
    admissions = df_stats_here['admissions'].to_numpy()
    all_x_lists.append(x_lists)
    all_admissions.append(admissions)

Starting SSNAP coefficients:

In [8]:
df_pop_admissions = pl.read_csv(os.path.join('outputs', 'ssnap_coeffs.csv'))

In [9]:
df_pop_admissions

Age Groups,population,prop_of_all_pop,count,prop_of_all_admissions,admissions_annual,admissions_annual_boost,prob_stroke_given_age
str,i64,f64,i64,f64,f64,f64,f64
"""Under 65""",45933245,0.816055,38827,0.231449,12942.33333,18737.66819,0.000408
"""65-69""",2796740,0.049687,15324,0.091347,5108.0,7395.26689,0.002644
"""70-74""",2779326,0.049378,21508,0.12821,7169.33333,10379.62674,0.003735
"""75-79""",1940686,0.034478,24150,0.143959,8050.0,11654.63948,0.006005
"""80 and over""",2836964,0.050402,67947,0.405035,22649.0,32790.7987,0.011558


Pick out SSNAP coefficients:

In [10]:
coeffs_ssnap = df_pop_admissions['prob_stroke_given_age'].to_numpy()

coeffs_ssnap

array([0.000408, 0.002644, 0.003735, 0.006005, 0.011558])

Pick out admissions numbers:

In [11]:
dict_admissions_age = dict(zip(df_pop_admissions['Age Groups'], df_pop_admissions['admissions_annual_boost']))

admissions_by_age = list(dict_admissions_age.values())

dict_admissions_age

{'Under 65': 18737.66819,
 '65-69': 7395.26689,
 '70-74': 10379.62674,
 '75-79': 11654.63948,
 '80 and over': 32790.7987}

## Gather starting coeffs

from best DEAP results

In [12]:
df_best_gens = pl.read_csv(os.path.join('outputs', 'best_inds_deap.csv'))

In [13]:
df_best_gens.sort('r2_all', descending=True)

dir,# gen,age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,fitness,r2_all,r2_q00,r2_q02,r2_q04,r2_q06,r2_q08
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""randomseed36""",26.0,1.276,1.232,1.202,1.2,1.236,1.1,1.1,1.1,1.1,1.108,0.901,1.0,1.0,1.0,1.0,0.9,0.9,1.0,0.9,0.897,0.8,0.9,0.8,0.8,0.886,238.973,0.595669,0.506855,0.580221,0.638502,0.614997,0.61723
"""randomseed95""",35.0,1.329,1.313,1.248,1.175,1.222,1.165,1.1,1.1,1.015,1.097,0.91,0.988,0.943,1.0,1.0,0.866,0.9,0.9,0.971,0.9,0.7,0.841,0.9,0.812,0.9,238.173,0.595636,0.505886,0.576829,0.638087,0.614718,0.622301
"""randomseed73""",48.0,1.3,1.3,1.218,1.2,1.2,1.0,1.111,1.1,1.106,1.12,0.992,0.971,1.0,0.95,1.0,0.897,0.971,0.973,0.9,0.9,0.8,0.803,0.8,0.83,0.9,237.915,0.595621,0.505944,0.58256,0.635348,0.61498,0.61906
"""randomseed52""",28.0,1.3,1.17,1.2,1.2,1.2,1.1,1.17,1.1,1.068,1.1,0.902,1.0,1.0,0.997,1.0,0.9,0.9,0.892,0.942,0.92,0.8,0.89,0.892,0.769,0.9,238.901,0.595597,0.503783,0.580277,0.638502,0.616117,0.618497
"""randomseed10""",26.0,1.299,1.359,1.267,1.2,1.2,0.994,1.068,1.144,1.1,1.1,0.932,1.0,1.0,1.0,1.0,0.932,0.9,0.9,0.9,0.9,0.851,0.831,0.8,0.8,0.9,238.809,0.595475,0.506986,0.581876,0.63755,0.613672,0.616852
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""randomseed09""",46.0,1.182,1.2,1.2,1.194,1.213,1.129,1.0,1.059,1.002,1.1,0.926,1.0,0.971,1.0,1.0,0.88,1.0,0.941,0.9,0.958,0.88,0.9,0.9,0.869,0.84,239.111,0.591768,0.494739,0.577935,0.637691,0.615445,0.611092
"""randomseed41""",35.0,1.2,1.2,1.21,1.164,1.2,1.1,1.162,1.161,1.1,1.122,1.0,0.98,1.0,0.962,1.0,0.9,0.941,0.935,0.9,0.9,0.785,0.85,0.8,0.845,0.9,239.308,0.59174,0.494242,0.575404,0.633268,0.614521,0.620178
"""randomseed82""",21.0,1.213,1.268,1.158,1.2,1.2,1.022,1.058,1.153,0.981,1.1,1.0,1.0,1.0,0.981,1.0,0.9,0.9,0.888,0.9,0.987,0.878,0.9,0.888,0.9,0.83,239.751,0.591289,0.496877,0.578808,0.632767,0.615709,0.611124


Pick out the directory names: 

In [14]:
best_dirs = df_best_gens.sort('r2_all', descending=True)['dir'].to_numpy()

Pick out the coefficients for each of these directories:

In [15]:
# Store start combos of coeffs in here:
coeffs_init_start = {}
scales_init_start = {}

for d, best_dir in enumerate(best_dirs):
    scales_init = df_best_gens.filter(df_best_gens['dir'] == best_dir)[coeff_names].to_numpy()[0]
    coeffs_init = [s for s in scales_init]
    for c, coeff in enumerate(coeffs_ssnap):
        for i in range(5):
            coeffs_init[5*i + c] *= coeff
    coeffs_init_start[best_dir] = coeffs_init
    scales_init_start[best_dir] = scales_init

# coeffs_init_grid = np.array(coeffs_init).reshape(5, 5)
# # To get all coeffs for one age band:        coeffs_init_grid[:, 0]
# # To get all coeffs for one depriv quantile: coeffs_init_grid[0, :]

Convert dictionary to dataframe:

In [16]:
df_coeffs_init = pl.concat([pl.DataFrame([c], schema=coeff_names) for c in coeffs_init_start.values()]).cast(float)

df_coeffs_init = df_coeffs_init.with_columns(pl.Series('coeff_combo', coeffs_init_start.keys()))#.cast(int))
# Move this column to start:
df_coeffs_init = df_coeffs_init.drop('coeff_combo').insert_column(0, df_coeffs_init.get_column('coeff_combo'))


/home/anna/miniconda3/envs/stroke_lsoa_prediction/lib/python3.10/functools.py:889: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  return dispatch(args[0].__class__)(*args, **kw)


Round results:

In [17]:
# labels = ['less65', '65', '70', '75', 'over80']
# round_dict = dict(zip(labels, [5, 4, 4, 4, 4]))

# for coeff in coeff_names:
#     key = coeff.split('_')[1]
#     new_data = np.round(df_coeffs_init[coeff], round_dict[key])
    
#     df_coeffs_init = df_coeffs_init.with_columns(pl.Series(coeff, new_data))

Check results:

In [18]:
df_coeffs_init

coeff_combo,age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""randomseed36""",0.000521,0.003257,0.004489,0.007206,0.014286,0.0004488,0.002908,0.0041085,0.0066055,0.012806,0.000368,0.002644,0.003735,0.006005,0.011558,0.0003672,0.0023796,0.003735,0.005405,0.010368,0.0003264,0.0023796,0.002988,0.004804,0.01024
"""randomseed95""",0.000542,0.003472,0.004661,0.007056,0.014124,0.000475,0.002908,0.0041085,0.006095,0.012679,0.000371,0.002612,0.003522,0.006005,0.011558,0.000353,0.0023796,0.0033615,0.005831,0.0104022,0.0002856,0.002224,0.0033615,0.004876,0.0104022
"""randomseed73""",0.0005304,0.0034372,0.004549,0.007206,0.01387,0.000408,0.002937,0.0041085,0.006642,0.012945,0.000405,0.002567,0.003735,0.005705,0.011558,0.000366,0.002567,0.003634,0.005405,0.0104022,0.0003264,0.002123,0.002988,0.004984,0.0104022
"""randomseed52""",0.0005304,0.003093,0.004482,0.007206,0.01387,0.0004488,0.003093,0.0041085,0.006413,0.0127138,0.000368,0.002644,0.003735,0.005987,0.011558,0.0003672,0.0023796,0.003332,0.005657,0.010633,0.0003264,0.002353,0.003332,0.004618,0.0104022
"""randomseed10""",0.00053,0.003593,0.004732,0.007206,0.01387,0.000406,0.002824,0.004273,0.0066055,0.0127138,0.00038,0.002644,0.003735,0.006005,0.011558,0.00038,0.0023796,0.0033615,0.005405,0.0104022,0.000347,0.002197,0.002988,0.004804,0.0104022
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""randomseed09""",0.000482,0.0031728,0.004482,0.00717,0.01402,0.000461,0.002644,0.003955,0.006017,0.0127138,0.000378,0.002644,0.003627,0.006005,0.011558,0.000359,0.002644,0.003515,0.005405,0.011073,0.000359,0.0023796,0.0033615,0.005218,0.009709
"""randomseed41""",0.0004896,0.0031728,0.004519,0.00699,0.01387,0.0004488,0.003072,0.004336,0.0066055,0.012968,0.000408,0.002591,0.003735,0.005777,0.011558,0.0003672,0.002488,0.003492,0.005405,0.0104022,0.00032,0.0022474,0.002988,0.005074,0.0104022
"""randomseed82""",0.000495,0.003353,0.004325,0.007206,0.01387,0.000417,0.002797,0.004306,0.005891,0.0127138,0.000408,0.002644,0.003735,0.005891,0.011558,0.0003672,0.0023796,0.003317,0.005405,0.011408,0.000358,0.0023796,0.003317,0.005405,0.009593


Save a copy:

In [19]:
df_coeffs_init.write_csv(os.path.join('genetic_algorithms', 'outputs', 'post_deap_inputs_for_opt.csv'))

## Function for optimising

In [20]:
def main_opt(coeffs, args):
    all_calculated_admissions = []
    all_observed_admissions = sum([list(a) for a in args[1]], [])

    
    # Convert scales to coefficients: IF NECESSARY --------------------------------
    coeffs_init = coeffs * np.concatenate([coeffs_ssnap] * 5)
    
    coeffs = np.round(coeffs, 7)


    x_lists = args[0]
    admissions_by_age = args[2]
    # coeffs_ssnap = args[3]
    # Predictions for each age band across England:
    predictions_by_age = [0.0] * 5
    for i in range(5):
        # Population numbers for areas in this quantile:
        x_lists_here = x_lists[i]
        # Separate prediction for each deprivation quantile:
        coeffs_here = coeffs[(i*5):(i*5)+5] #* np.array(coeffs_ssnap)
        yhat_list = predict_admissions_each_age(x_lists_here, coeffs_here)
        # predictions_lists.append(yhat)
        for j, y in enumerate(yhat_list):
            predictions_by_age[j] += y
    # Divide the admissions by age band by the observed values:
    for j in range(len(predictions_by_age)):
        predictions_by_age[j] /= admissions_by_age[j]
    # Wrongness ratio factor:
    rat = sum(np.abs(np.array(predictions_by_age) - 1.0)) + 1.0

    
    coeffs_grid = np.array(coeffs).reshape(5, 5)
    for i, q in enumerate(qmin_list):
        coeffs_q = coeffs_grid[i, :]
        x_lists = args[0][i]
        admissions = args[1][i]
        
        calculated_admissions = predict_admissions(x_lists, np.array(coeffs_q))

        all_calculated_admissions += list(calculated_admissions)

    # Calculate measure of difference from the observed admissions:
    # print(
    #     np.array(all_calculated_admissions),
    #     np.array(all_observed_admissions)
    # )
    # diff = find_mean_abs_diff(
    diff = find_square_residuals(
        np.array(all_calculated_admissions),
        np.array(all_observed_admissions)
    )
    # Apply wrongness factor:
    diff *= rat

    
    diff_before = diff
    n_penalty = 0
    use_penalty = True
    if use_penalty:
        # Check coefficient conditions. If any aren't met, add a penalty
        # to the main difference measure.
        n_decrease = 0.0
        for i, q in enumerate(qmin_list):
            # Pick out coeffs for this depriv quantile:
            coeffs_q = coeffs_grid[i, :]
            # Do all coeffs increase with age band?
            n_decrease += sum(np.sign(np.diff(coeffs_q)) < 0)

        n_increase = 0.0
        for i, a in enumerate(age_numbers):
            # Pick out coeffs for this age band:
            coeffs_q = coeffs_grid[:, i]
            # Do all coeffs decrease with depriv quantile?
            n_increase += sum(np.sign(np.diff(coeffs_q)) > 0)

        # Add penalty:
        n_penalty = n_increase + n_decrease
        if n_penalty > 0.0:
            diff += (n_penalty * abs(diff) * 0.2)
    # print(diff_before, n_penalty, diff)
    # print(coeffs_grid)
    # # print(n_decrease, n_increase, diff)
    # print('\n' * 3)

    return diff

In [21]:
def predict_admissions(x_lists, coeffs):
    """
    x_lists: np.array.
    coeffs: np.array.
    
    Have to have same number of coeffs as x_lists.
    """
    # Predicted admissions:
    yhat = (x_lists * coeffs.reshape(len(coeffs), 1)).sum(axis=0)
    return yhat

In [22]:
def predict_admissions_each_age(x_lists, coeffs):
    """
    x_lists: np.array.
    coeffs: np.array.
    
    Have to have same number of coeffs as x_lists.
    """
    # Predicted admissions:
    # yhat = (x_lists * coeffs.reshape(len(coeffs), 1)).sum(axis=0)
    # yhat = [(x_lists[i] * coeffs[i]).to_numpy() for i in range(len(coeffs))]
    yhat = (
        [(x_lists[i] * coeffs[i]).sum() for i in range(len(coeffs))]
    )
    return yhat

Goodness check option 1: This calculates the sum of the square of the differences between predicted and actual admission numbers:

In [23]:
def find_square_residuals(yhat, y):
    # Difference from actual:
    sqres = (yhat - y)**2.0
    # Sum of differences:
    sum_sqres = np.sqrt(sqres.sum())
    return sum_sqres

Goodness check option 2: This calculates the mean absolute difference between the predicted and real admissions numbers:

In [24]:
def find_mean_abs_diff(yhat, y):
    # Difference from actual:
    absres = np.abs((yhat - y))
    # Mean of differences:
    mean_absres = absres.mean()
    return mean_absres

To check accuracy, the following function calculates R-squared:

In [25]:
def calculate_rsquared(y, yhat):
    """This gives the same results as the sklearn built-in."""
    y_mean = y.mean()
    ss_res = ((yhat - y)**2.0).sum()
    ss_tot = ((y - y_mean)**2.0).sum()
    if ss_tot != 0.0:
        rsq = 1.0 - ss_res / ss_tot
    else:
        rsq = np.NaN
    return rsq

## Run grid

In [151]:
best_dirs = df_best_gens.filter(np.round(df_best_gens['r2_all'], 3) == np.round(df_best_gens['r2_all'].max(), 3))['dir'].to_numpy()

# Only run these best starting coeffs:
coeffs_init_start_opt = {}

for k, v in coeffs_init_start.items():
    if k in best_dirs:
        coeffs_init_start_opt[k] = v
    else:
        pass

In [152]:
dict_coeffs = {}
dict_opt_results = {}
# Store these opt results keys:
opt_results_keys = ['message', 'success', 'status', 'nit', 'nfev']

i = 1
for coeff_combo, coeffs_init in coeffs_init_start_opt.items():
    print(f'{i:5d} out of {len(coeffs_init_start_opt)}', end='\r')
    i += 1
    bounds_here = [(0.0, 1.0)] * len(coeffs_init)  # force results to lie between 0 and 1
    
        
    # Run the optimiser:
    opt_results = minimize(
        main_opt,
        x0=coeffs_init,
        args=[all_x_lists, all_admissions, admissions_by_age],
        bounds=bounds_here,
        method='Nelder-Mead',
        # method='SLSQP',
        options=dict(maxiter=10000),
    )
    # Pick out the resulting health coefficients:
    coeffs = np.round(opt_results['x'], 7)

    # Calculate R^2 here:

    # Log whether this combo optimised successfully:
    dict_opt_results[coeff_combo] = pl.DataFrame(np.array([[f'{coeff_combo}'] + [opt_results[k] for k in opt_results_keys]]), schema=['coeff_combo'] + opt_results_keys)
    # pl.Series(f'{coeff_combo}', [opt_results[k] for k in opt_results_keys], strict=False).to_frame()
    # Store coeffs:
    dict_coeffs[coeff_combo] = pl.DataFrame(np.array([[f'{coeff_combo}'] + list(coeffs)]), schema=['coeff_combo'] + coeff_names)
    # pl.Series(f'{coeff_combo}', coeffs).to_frame()

KeyboardInterrupt: 

Gather results into dataFrames:

In [ ]:
df_opt_results = pl.concat(dict_opt_results.values())

df_coeffs = pl.concat(dict_coeffs.values())#.cast(float)
# df_coeffs = df_coeffs.with_columns(pl.Series('coeff_combo', df_coeffs['coeff_combo'].cast(int)))
for c in coeff_names:
    df_coeffs = df_coeffs.with_columns(pl.Series(c, df_coeffs[c].cast(float)))

df_results = df_opt_results.join(df_coeffs, on='coeff_combo', how='left')

Calculate r-squared values of each fit:

In [ ]:
list_r2_dicts = []

for coeff_combo in df_results['coeff_combo']:
    dict_r2 = {}
    mask = df_results['coeff_combo'] == coeff_combo
    row_coeffs = df_results.filter(mask)

    df_admissions_here = df_stats[['MSOA', 'depriv_quantile_min', 'admissions'] + age_numbers]
    df_admissions_here = df_admissions_here.with_columns(pl.Series('admissions_predicted', np.zeros(len(df_admissions_here))))
    
    for q in qmin_list:
        qstr = str(round(q, 1)).replace('.', '')
        df_here = df_admissions_here.filter(df_admissions_here['depriv_quantile_min'] == q)
        
        for a in age_numbers:
            df_here = df_here.with_columns(pl.Series('admissions_predicted', df_here['admissions_predicted'] + (row_coeffs[f'{a}_q{qstr}'].to_numpy()[0] * df_here[a])))

        # R2 for just this q:
        r2_q = calculate_rsquared(df_here['admissions'], df_here['admissions_predicted'])
        dict_r2[f'r2_q{qstr}'] = np.round(r2_q, 5)
        
        mask = df_admissions_here['MSOA'].is_in(df_here['MSOA'])

        # df_admissions_here = df_admissions_here.with_columns(
        #    pl.when(mask)
        #      .then(df_here['admissions_predicted'])
        #      .otherwise(pl.col('admissions_predicted'))
        #      .name.keep()
        # )
        df_admissions_here = df_admissions_here.join(df_here['MSOA', 'admissions_predicted'], on='MSOA', suffix=f'_q{qstr}', how='left')
        
    cols_pred = [c for c in df_admissions_here.columns if (('q' in c) & (c.startswith('admissions_predicted')))]
    df_admissions_here = df_admissions_here.with_columns(pl.Series('admissions_predicted', df_admissions_here[cols_pred].sum_horizontal()))
    
    # Calculate R2:
    r2 = calculate_rsquared(df_admissions_here['admissions'], df_admissions_here['admissions_predicted'])
    dict_r2['r2_all'] = np.round(r2, 5)
    # Calculate residuals:
    res = find_square_residuals(df_admissions_here['admissions_predicted'], df_admissions_here['admissions'])
    dict_r2['res'] = np.round(res, 3)
            

    list_r2_dicts.append(pl.DataFrame([[coeff_combo] + list(dict_r2.values())], schema=['coeff_combo'] + list(dict_r2.keys())))

Convert results to dataframe:

In [ ]:
df_r2 = pl.concat(list_r2_dicts, how='vertical')

df_r2.sort('res')

Merge R2 results into main optimisation results:

In [ ]:
df_results_r2 = df_results.join(df_r2, on='coeff_combo', how='left')

In [ ]:
df_results_r2

In [ ]:
df_results_r2.write_csv(os.path.join('genetic_algorithms', 'outputs', 'post_deap_multi_optimise_coeffs_results.csv'))

## Annealing

Code to overwrite coeffs if only some combos should be run:

Run the annealing:

In [26]:
scales_init

array([1.2  , 1.2  , 1.2  , 1.1  , 1.304, 1.   , 1.1  , 1.1  , 1.037,
       1.092, 1.   , 1.026, 1.   , 1.   , 1.   , 0.9  , 0.9  , 0.9  ,
       0.9  , 0.9  , 0.9  , 0.9  , 0.883, 0.9  , 0.87 ])

In [27]:
dict_coeffs = {}
dict_opt_results = {}
# Store these opt results keys:
opt_results_keys = ['success', 'nit', 'minimization_failures', 'nfev']  # basin-hopping

i = 1
for coeff_combo, coeffs_init in coeffs_init_start.items():
# for coeff_combo, scales_init in scales_init_start.items():
    print(f'{i:5d} out of {len(coeffs_init_start)}', end='\r')
    # print(f'{i:5d} out of {len(scales_init_start)}', end='\r')
    i += 1

    # # Convert scales to coefficients:
    # coeffs_init = scales_init * np.concatenate([coeffs_ssnap] * 5)
    
    bounds_here = [(0.0, 1.0)] * len(coeffs_init)  # force results to lie between 0 and 1
    # bounds_here = [(0.0, 2.5)] * len(scales_init)  # force results to lie between 0 and 1
    
    # Run the optimiser:
    opt_results = basinhopping(
        main_opt,
        x0=coeffs_init,  # scales_init,  # coeffs_init,
        T=10.0,  # experiment
        stepsize=5e-5,  # 5e-5,
        minimizer_kwargs = dict(
            args=[all_x_lists, all_admissions, admissions_by_age],
            bounds=bounds_here,
            method='Nelder-Mead',
            options=dict(maxiter=10000),
            ),
    )
    # Pick out the resulting health coefficients:
    coeffs = np.round(opt_results['x'], 7)

    # Calculate R^2 here:

    # Log whether this combo optimised successfully:
    opt_results_to_save = [opt_results['message'][0]] + [opt_results[k] for k in opt_results_keys]
    # Log whether this combo optimised successfully:
    dict_opt_results[coeff_combo] = pl.DataFrame(np.array([[f'{coeff_combo}'] + opt_results_to_save]), schema=['coeff_combo', 'message'] + opt_results_keys)  # basin-hopping
    # pl.Series(f'{coeff_combo}', [opt_results[k] for k in opt_results_keys], strict=False).to_frame()
    # Store coeffs:
    dict_coeffs[coeff_combo] = pl.DataFrame(np.array([[f'{coeff_combo}'] + list(coeffs)]), schema=['coeff_combo'] + coeff_names)
    # pl.Series(f'{coeff_combo}', coeffs).to_frame()

/home/anna/miniconda3/envs/stroke_lsoa_prediction/lib/python3.10/site-packages/scipy/optimize/_basinhopping.py:294: OptimizeWarning: Initial guess is not within the specified bounds
  return self.minimizer(self.func, x0, **self.kwargs)


Gather results into dataFrames:

In [28]:
df_ann_opt_results = pl.concat(dict_opt_results.values())

df_ann_coeffs = pl.concat(dict_coeffs.values())
for c in coeff_names:
    df_ann_coeffs = df_ann_coeffs.with_columns(pl.Series(c, df_ann_coeffs[c].cast(float)))

df_ann_results = df_ann_opt_results.join(df_ann_coeffs, on='coeff_combo', how='left')

Calculate r-squared of results:

In [29]:
list_r2_dicts = []

for coeff_combo in df_ann_coeffs['coeff_combo']:
    dict_r2 = {}
    mask = df_ann_coeffs['coeff_combo'] == coeff_combo
    row_coeffs = df_ann_coeffs.filter(mask)

    df_admissions_here = df_stats[['MSOA', 'depriv_quantile_min', 'admissions'] + age_numbers]
    df_admissions_here = df_admissions_here.with_columns(pl.Series('admissions_predicted', np.zeros(len(df_admissions_here))))
    
    for q in qmin_list:
        qstr = str(round(q, 1)).replace('.', '')
        df_here = df_admissions_here.filter(df_admissions_here['depriv_quantile_min'] == q)
        
        for a in age_numbers:
            df_here = df_here.with_columns(pl.Series('admissions_predicted', df_here['admissions_predicted'] + (row_coeffs[f'{a}_q{qstr}'].to_numpy()[0] * df_here[a])))
            # df_here = df_here.with_columns(pl.Series('admissions_predicted', df_here['admissions_predicted'] + (row_coeffs[f'{a}_q{qstr}'].to_numpy()[0] * df_here[a])))

        # R2 for just this q:
        r2_q = calculate_rsquared(df_here['admissions'], df_here['admissions_predicted'])
        dict_r2[f'r2_q{qstr}'] = np.round(r2_q, 5)
        
        mask = df_admissions_here['MSOA'].is_in(df_here['MSOA'])

        # df_admissions_here = df_admissions_here.with_columns(
        #    pl.when(mask)
        #      .then(df_here['admissions_predicted'])
        #      .otherwise(pl.col('admissions_predicted'))
        #      .name.keep()
        # )
        df_admissions_here = df_admissions_here.join(df_here['MSOA', 'admissions_predicted'], on='MSOA', suffix=f'_q{qstr}', how='left')
        
    cols_pred = [c for c in df_admissions_here.columns if (('q' in c) & (c.startswith('admissions_predicted')))]
    df_admissions_here = df_admissions_here.with_columns(pl.Series('admissions_predicted', df_admissions_here[cols_pred].sum_horizontal()))
    
    # Calculate R2:
    r2 = calculate_rsquared(df_admissions_here['admissions'], df_admissions_here['admissions_predicted'])
    dict_r2['r2_all'] = np.round(r2, 5)
    # Calculate residuals:
    res = find_square_residuals(df_admissions_here['admissions_predicted'], df_admissions_here['admissions'])
    dict_r2['res'] = np.round(res, 3)
            

    list_r2_dicts.append(pl.DataFrame([[coeff_combo] + list(dict_r2.values())], schema=['coeff_combo'] + list(dict_r2.keys())))

Convert results to dataframe:

In [30]:
df_ann_r2 = pl.concat(list_r2_dicts, how='vertical')

Merge R2 results into main optimisation results:

In [31]:
df_ann_results = df_ann_results.join(df_ann_r2, on='coeff_combo', how='left')

In [32]:
df_ann_results

coeff_combo,message,success,nit,minimization_failures,nfev,age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,r2_q00,r2_q02,r2_q04,r2_q06,r2_q08,r2_all,res
str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""randomseed36""","""requested number of basinhoppi…","""True""","""100""","""0""","""114337""",0.0005627,0.0032759,0.0046295,0.006953,0.0140609,0.0004411,0.003041,0.0043766,0.0060557,0.0126583,0.000392,0.0024991,0.0039313,0.0058089,0.0111146,0.0003657,0.0024164,0.0031797,0.0055187,0.0107382,0.0002659,0.0023602,0.0029913,0.0053872,0.010532,0.50362,0.58139,0.63463,0.61611,0.62451,0.59608,236.695
"""randomseed95""","""requested number of basinhoppi…","""True""","""100""","""0""","""113081""",0.0005156,0.0034349,0.0046684,0.0069729,0.0141363,0.0004708,0.0029352,0.0041158,0.0061496,0.0125898,0.0003817,0.0025813,0.0035292,0.0060128,0.0116126,0.0003562,0.0024018,0.0033917,0.0058241,0.0105346,0.0003075,0.0022643,0.0033213,0.0048203,0.0103005,0.50695,0.57837,0.6376,0.61639,0.62035,0.59599,236.722
"""randomseed73""","""requested number of basinhoppi…","""True""","""100""","""0""","""113476""",0.0005673,0.0035611,0.0047694,0.0075481,0.0129085,0.0004525,0.0029716,0.0040759,0.0063787,0.0120755,0.0003589,0.0026055,0.0035683,0.0056083,0.0119638,0.000351,0.0022893,0.0034876,0.0055548,0.0110996,0.0002973,0.0022297,0.0031554,0.0049381,0.010548,0.5031,0.57777,0.63977,0.61842,0.62263,0.59651,236.57
"""randomseed52""","""requested number of basinhoppi…","""True""","""100""","""0""","""118319""",0.0005404,0.0030518,0.0052983,0.0081198,0.0127288,0.0004446,0.0029753,0.0036516,0.0072165,0.0123751,0.0003616,0.0029499,0.0035652,0.0061631,0.0113087,0.0003501,0.0027023,0.0034184,0.004936,0.0110964,0.0003354,0.0018535,0.0032153,0.0040981,0.0110095,0.50499,0.58043,0.6381,0.6181,0.62087,0.59661,236.54
"""randomseed10""","""requested number of basinhoppi…","""True""","""100""","""0""","""111323""",0.0005348,0.0033393,0.0044116,0.0089197,0.012812,0.0004611,0.0029972,0.0040739,0.0067606,0.0122731,0.0003641,0.0026068,0.0037764,0.006193,0.0114576,0.0003629,0.0025289,0.0035098,0.0045183,0.0112826,0.000308,0.0021287,0.0031691,0.0043414,0.0107261,0.5046,0.57912,0.6386,0.61802,0.62084,0.59636,236.612
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""randomseed09""","""requested number of basinhoppi…","""True""","""100""","""0""","""118032""",0.0005808,0.0028522,0.0062296,0.0066828,0.0125079,0.0004906,0.0028066,0.0037888,0.0065605,0.0123111,0.0003416,0.0027845,0.0034361,0.0064707,0.0115995,0.0003286,0.0027501,0.0033585,0.0051114,0.01107,0.0002794,0.0022601,0.0027023,0.0049895,0.0109591,0.50224,0.57577,0.64034,0.61852,0.62473,0.5965,236.572
"""randomseed41""","""requested number of basinhoppi…","""True""","""100""","""0""","""120179""",0.0005344,0.0036845,0.0049691,0.0073817,0.0130876,0.0004842,0.0031195,0.0042166,0.0062504,0.0119951,0.0003892,0.0025051,0.0037697,0.0058757,0.0113169,0.0003316,0.0022545,0.0032782,0.0055054,0.0112789,0.0002873,0.0021439,0.0029478,0.0049409,0.0108887,0.50626,0.57608,0.63577,0.61962,0.62475,0.59649,236.574
"""randomseed82""","""requested number of basinhoppi…","""True""","""100""","""0""","""112946""",0.0005292,0.0032703,0.0053996,0.0072186,0.0136222,0.0004734,0.0030888,0.0038673,0.0061317,0.0124478,0.0003868,0.002794,0.0034018,0.0057412,0.0115017,0.0003379,0.002536,0.003321,0.0055244,0.0109416,0.0003017,0.0019144,0.0032365,0.0052273,0.010413,0.50707,0.57791,0.63622,0.61834,0.62164,0.59625,236.645


Save results:

In [33]:
df_ann_results.write_csv(os.path.join('genetic_algorithms', 'outputs', 'post_deap_multi_basinhop_coeffs_results.csv'))